# Document

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="Employee receive 20 days of annual leave per year.",
    metadata={
        "source": "leave_policy.pdf",
        "department": "HR",
        "page": 5,
    },
)

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Employees receive 20 days of annual leave.",
        metadata={"source": "leave_policy.pdf"},
    ),
    Document(
        page_content="Employees must request annual leave through HR",
        metadata={"source": "leave_policy.pdf"},
    ),
]

for document in documents:
    print(document.page_content)
    print(document.metadata)

Employees receive 20 days of annual leave.
{'source': 'leave_policy.pdf'}
Employees must request annual leave through HR
{'source': 'leave_policy.pdf'}


## Real Enterprise Example

In [ ]:
Document(
    page_content="Employees receive 20 days annual leave.",
    metadata={
        "source": "leave_policy.pdf",
        "page": 5,
        "department": "HR",
        "version": "2026.1",
    },
)

Document(metadata={'source': 'leave_policy.pdf', 'page': 5, 'department': 'HR', 'version': '2026.1'}, page_content='Employees receive 20 days annual leave.')

# Embedding

## LangChain Embedding

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector = embeddings.embed_query("How many days of annual leave do employees get?")

print(len(vector))

1536


# VectorStore

In [ ]:
from config import EMBEDDING_MODEL
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

documents = [
    Document(page_content="Employees receive 20 days annual leave."),
    Document(page_content="Employees must use MFA for company systems."),
]

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents)

result = vector_store.similarity_search(
    "How much annual leave do employees receive?",
    k=2,
)

for doc in result:
    print(doc.page_content)

Employees receive 20 days annual leave.
Employees must use MFA for company systems.


# Prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
You are an internal company knowledge assistant.

Use ONLY the context below.

Context:
{context}

Question:
{question}

Instructions:
- Do not invent information.
- Give a concise answer.
- Include the source document when available.
- If the answer is not in the context, say:
"I could not find this information in the company knowledge base."
"""
)

# LLM

In [ ]:
from config import CHAT_MODEL
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

response = llm.invoke("Explain what RAG is in simple English.")

print(response.content)

RAG stands for **Retrieval-Augmented Generation**.

In simple English, it means:

1. **The AI looks up information first**  
   Instead of answering only from what it remembers, it searches a database, documents, or the web for relevant info.

2. **Then it writes the answer**  
   It uses that retrieved information to generate a response.

### Why this is useful
- **More accurate**: It can use up-to-date or specific information.
- **Less guessing**: It doesn’t have to rely only on memory.
- **Good for private data**: It can answer questions from your company’s documents.

### Simple example
If you ask, “What is our refund policy?”  
A RAG system might:
- search your policy documents,
- find the right section,
- then explain it in plain language.

So, RAG is basically **“look it up, then answer.”**


# Chain

In [ ]:
from config import CHAT_MODEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template("Explain {topic} in simple English.")

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
)

chain = prompt | llm

result = chain.invoke({"topic": "RAG"})

print(result.content)

RAG stands for **Retrieval-Augmented Generation**.

In simple English, it means:

1. **Retrieve**: First, the AI looks up relevant information from a source like documents, files, or a database.
2. **Augment**: It adds that found information to the prompt or context.
3. **Generate**: Then the AI writes an answer using both its own language ability and the retrieved info.

### Why use RAG?
Because large language models don’t always know the latest facts and can sometimes make things up. RAG helps by letting the AI **check trusted information before answering**.

### Example
If you ask, “What is our company’s refund policy?”  
A RAG system might:
- search the company handbook,
- find the refund section,
- then answer your question based on that text.

### In short
**RAG = AI that searches for useful info first, then uses it to answer better.**

If you want, I can also explain it with a **real-life analogy** or a **diagram-style example**.


# Tool

In [ ]:
from langchain.tools import tool


@tool
def get_leave_balance(employee_id: str) -> str:
    """Return the employee's current annual leave balance"""

    # In production, call the HR system.
    return "Employee has 12.5 days of annual leave remaining."

# Agent

In [10]:
from config import CHAT_MODEL
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"The Weather in {city} is sunny."


model = ChatOpenAI(model=CHAT_MODEL, temperature=0)

agent = create_agent(model=CHAT_MODEL, tools=[get_weather])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Sydney?"}]}
)

print(result)

{'messages': [HumanMessage(content='What is the weather in Sydney?', additional_kwargs={}, response_metadata={}, id='e6146d9d-c654-4bfa-9060-ab68c29e16bb'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 133, 'total_tokens': 150, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EW9PN1IBkjIG5DPE2NFiY2KCKVbo4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a113ab-8133-7011-980b-5045210404e3-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Sydney'}, 'id': 'call_t5acziW8DvRmyXKn0YZOCZ2y', 'type